In [5]:
import os, re, json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
solar_api = os.getenv("SOLAR_API")

client = OpenAI(
    api_key=solar_api,
    base_url="https://api.upstage.ai/v1"
)


In [6]:
content = """

2. 주요 제품 및 서비스
가. 주요제품등의 현황
(단위 : 백만원)
매출유형
품목
2024년도(제25기)
2023년도(제24기)
2022년도(제23기)
금액
비중
금액
비중
금액
비중
제품
큐피스템줄기세포화장품 등
512
7.4%
1,142
17.6%
657
10.0%
상품
레모둘린 등
6,419
92.6%
5,356
82.4%
5,934
90.0%
기타(용역/기술 매출 등)
2
0.0%
2
0.0%
-
-
합계
6,933
100.0%
6,500
100.0%
6,591
100.0%
나. 주요 제품등의 가격변동 추이
당사가 생산하는 제품의 판매가격은 고객과의 계약에 따라 결정된 최초 판매가격과 계약에서 정한 고정비율만큼 상승하는 등의 계약조건에 따라 조정될 수 있습니다.
"""

In [7]:
prompt = """
너는 기업 사업보고서의 "주요 제품", "주요 제품 및 서비스",
또는 이와 의미가 동일한 표에서 매출 구성 정보를 추출한다.

기업마다 표 구조가 다르므로 표를 먼저 해석한 뒤 계산한다.

────────────────
[1단계 — 표 구조 파악]
────────────────

표에서 다음 요소를 식별한다.

① 기수 또는 기간 (예: 제23기, 2023년 등)
② 품목 또는 제품명
③ 매출 금액 또는 매출 비율
④ 합계 행

표 형태는 기업마다 다르므로
열 이름, 순서, 병합 여부에 의존하지 않는다.

────────────────
[2단계 — 값 유형 판별]
────────────────

표가 다음 중 무엇인지 판별한다.

CASE A: 비율표
→ 각 항목에 이미 % 값 있음

CASE B: 금액표
→ 금액만 있고 합계 존재

CASE C: 금액 + 비율 혼합표
→ 비율 값 우선 사용

────────────────
[3단계 — 값 추출 규칙]
────────────────

품목:
- 같은 셀의 줄바꿈 텍스트는 하나의 품목
- 셀을 분리하지 않는다

금액:
- 쉼표 제거 후 숫자 사용
- 공란, '-' 는 0

비율:
- 숫자만 추출 후 %로 변환

합계:
- 합계 / 총계 / 계
- 각 기간별 존재 가능

────────────────
[4단계 — 계산 규칙]
────────────────

CASE A (비율표)
→ 그대로 사용

CASE B (금액표)
→ 비중 = 금액 / 합계 × 100

CASE C (혼합표)
→ 비율 값 사용

소수 둘째 자리 반올림

────────────────
[5단계 — 기간 독립성]
────────────────

각 기간은 독립 계산
다른 기간 값 사용 금지

────────────────
[6단계 — 검증]
────────────────

항목 비율 합 = 100%

초과 또는 부족 시
합계 또는 값 재확인 후 수정

────────────────
[7단계 — 출력]
────────────────

JSON만 출력
설명 금지

{
  "<기간>": {
    "<품목>": "<비중%>",
    ...
    "합계": "100.00%"
  }
}

기간이 여러 개면 모두 출력
기간 정보 없으면 "기간미상"
"""


In [8]:
import json, re
from IPython.display import display, JSON

# result → JSON 안전 추출
match = re.search(r"\{.*\}", result, re.S)
if not match:
    raise ValueError("LLM 응답에서 JSON 못 찾음")

data = json.loads(match.group())

# -------------------------
# 기간 키 자동 탐색
# -------------------------
period_key = None
for k in ["기간", "기수", "연도", "사업연도"]:
    if k in data:
        period_key = k
        break

if period_key is None:
    raise KeyError(f"기간 키 없음. 현재 키: {list(data.keys())}")

periods = data[period_key]

# -------------------------
# 최신기 index
# -------------------------
def period_num(p):
    m = re.search(r"\d+", str(p))
    return int(m.group()) if m else -1

idx = max(range(len(periods)), key=lambda i: period_num(periods[i]))
latest_period = periods[idx]

# -------------------------
# 숫자 변환 함수 (쉼표/문자 대비)
# -------------------------
def to_num(x):
    if isinstance(x, (int, float)):
        return x
    if x in ("", "-", None):
        return 0
    return float(re.sub(r"[^\d\-\.]", "", str(x)) or 0)

# -------------------------
# 합계
# -------------------------
total = None
if "합계" in data:
    total = to_num(data["합계"][idx])

# 합계 없거나 0이면 직접 계산
if not total:
    total = sum(
        to_num(v[idx])
        for k, v in data.items()
        if k not in (period_key, "합계")
    )
    print(f"⚠ 합계 미검출 → 품목 합 사용: {total}")

if total == 0:
    raise ValueError("합계 0 → 금액 추출 실패")

# -------------------------
# 비중 계산
# -------------------------
ratio = {}
for k, v in data.items():
    if k in (period_key, "합계"):
        continue
    ratio[k] = f"{(to_num(v[idx]) / total * 100):.2f}%"

ratio["합계"] = "100.00%"

latest_only = {latest_period: ratio}

# -------------------------
# 출력
# -------------------------
print(json.dumps(latest_only, indent=2, ensure_ascii=False))
display(JSON(latest_only))


NameError: name 'result' is not defined

In [ ]:

final_input = f"""
다음 문서에서 주요 제품별 매출 비중을 추출하라.

문서:
{content}

지침:
{prompt}
"""


In [ ]:
messages = [
    {"role": "user", "content": final_input}
]


In [ ]:

response = client.chat.completions.create(
    model="solar-1-mini-chat",   
    messages=messages,
    temperature=0
)

result = response.choices[0].message.content
print("✅ result 생성 완료")


✅ result 생성 완료


In [ ]:
print("===== LLM AMOUNT DATA =====")
print(json.dumps(amount_data, indent=2, ensure_ascii=False))


===== LLM AMOUNT DATA =====
{
  "제23기": {
    "큐피스템, 줄기세포배양액 등": 5934,
    "레모둘린 등": 657,
    "기타(용역/기술 매출 등)": 1089,
    "합계": 7679
  }
}


In [ ]:
import re, json
from IPython.display import display, JSON

m = re.search(r"\{.*\}", result, flags=re.S)
amount_data = json.loads(m.group())

def _period_num(k: str):
    mm = re.search(r"제\s*(\d+)\s*기", k)
    return int(mm.group(1)) if mm else -1

latest_period = max(amount_data.keys(), key=_period_num)
items = amount_data[latest_period]

# ⭐ 합계 처리 (핵심 수정)
total = items.get("합계")
if not total or total == 0:
    total = sum(v for k, v in items.items() if k != "합계")
    print(f"⚠ 합계 미검출 → 품목 합으로 계산: {total}")

# 비중 계산
ratio = {}
for k, v in items.items():
    if k == "합계":
        continue
    ratio[k] = f"{(v/total*100):.2f}%"

ratio["합계"] = "100.00%"

latest_only = {latest_period: ratio}

print("===== LATEST ONLY (CALCULATED) =====")
print(json.dumps(latest_only, indent=2, ensure_ascii=False))
display(JSON(latest_only))


===== LATEST ONLY (CALCULATED) =====
{
  "제23기": {
    "큐피스템, 줄기세포배양액 등": "77.28%",
    "레모둘린 등": "8.56%",
    "기타(용역/기술 매출 등)": "14.18%",
    "합계": "100.00%"
  }
}


<IPython.core.display.JSON object>

In [ ]:
#json파싱 검증
import json

try:
    data = json.loads(result)
except json.JSONDecodeError:
    print("⚠ JSON 파싱 실패 → 재요청 필요")

#합계 100 검증
def check_total(data):
    for period, items in data.items():
        if "합계" in items:
            total = float(items["합계"].replace("%",""))
            if abs(total - 100) > 0.1:
                print(f"⚠ 합계 이상: {period}")

#빈 결과 방지
if not data:
    print("⚠ 주요 제품 매출 비중 추출 실패")
